In [1]:

%pip install -q xgboost scikit-learn pandas numpy

In [2]:

import os
import time
import numpy as np
import pandas as pd
import sklearn
import xgboost

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

print("Setup successful!")
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)
print("XGBoost:", xgboost.__version__)
print("Random Forest: ready")

Setup successful!
Pandas: 2.2.3
Scikit-learn: 1.6.1
XGBoost: 3.4.1
Random Forest: ready


In [3]:

from google.colab import files

uploaded = files.upload()


Saving observations.csv to observations.csv


In [4]:

from google.colab import files

uploaded = files.upload()



Saving Student folder.zip to Student folder.zip
Saving Student folder-20261118T050914Z-1-001.zip to Student folder-20261118T050914Z-1-001.zip
Saving candidates.csv to candidates.csv
Saving observations.csv to observations (1).csv


In [5]:

import os

for name in [
    "candidates.csv",
    "observations.csv",
    "Student folder (1).zip"
]:
    if os.path.exists(name):
        print("FOUND:", name)
    else:
        print("MISSING:", name)


FOUND: candidates.csv
FOUND: observations.csv
MISSING: Student folder (1).zip


In [6]:

import pandas as pd
import os

candidates = pd.read_csv("candidates.csv")
observations = pd.read_csv("observations.csv")

print("CANDIDATES SHAPE:", candidates.shape)
print("\nCANDIDATES COLUMNS:")
print(candidates.columns.tolist())
print("\nFIRST 5 CANDIDATE ROWS:")
display(candidates.head())

print("\nOBSERVATIONS SHAPE:", observations.shape)
print("\nOBSERVATIONS COLUMNS:")
print(observations.columns.tolist())
print("\nFIRST 5 OBSERVATION ROWS:")
display(observations.head())



CANDIDATES SHAPE: (1198560, 5)

CANDIDATES COLUMNS:
['trip_id', 'obs_id', 'edge_id', 'distance_m', 'distance_rank']

FIRST 5 CANDIDATE ROWS:


,trip_id,obs_id,edge_id,distance_m,distance_rank
0,1,1,64973,0.722768,1
1,1,1,117639,15.132321,2
2,1,1,2641,32.198445,3
3,1,10,139208,5.812021,1
4,1,10,133950,8.442955,2



OBSERVATIONS SHAPE: (227577, 4)

OBSERVATIONS COLUMNS:
['trip_id', 'obs_id', 'ref_edge_id', 'split']

FIRST 5 OBSERVATION ROWS:


,trip_id,obs_id,ref_edge_id,split
0,1,1,64973,train
1,1,2,64973,train
2,1,3,64973,train
3,1,4,64973,train
4,1,5,64973,train


In [10]:

import os
import zipfile
import time
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from xgboost import XGBClassifier
from google.colab import files

# --------------------------------------------------
# 1. Check input files
required = [
    "candidates.csv",
    "observations.csv",
    "Student folder.zip"
]


missing = [f for f in required if not os.path.exists(f)]
if missing:
    raise FileNotFoundError(
        f"Missing files: {missing}. Upload them in Step 1 and rerun."
    )

print("Loading CSV files...")
candidates = pd.read_csv("candidates.csv")
observations = pd.read_csv("observations.csv")

print("Candidates:", candidates.shape)
print("Observations:", observations.shape)
print("Split counts:")
print(observations["split"].value_counts())

# --------------------------------------------------
# 2. Create labels and candidate features
# --------------------------------------------------
# A candidate is positive when its edge_id matches
# the reference edge for that observation.
data = candidates.merge(
    observations[["trip_id", "obs_id", "ref_edge_id", "split"]],
    on=["trip_id", "obs_id"],
    how="inner",
    validate="many_to_one"
)

data["label"] = (
    data["edge_id"].astype(str) ==
    data["ref_edge_id"].astype(str)
).astype(int)

# Features use candidate distances/ranks only.
# No reference edge or label is used as a model feature.
data["distance_m"] = pd.to_numeric(
    data["distance_m"], errors="coerce"
)
data["distance_rank"] = pd.to_numeric(
    data["distance_rank"], errors="coerce"
)

group_cols = ["trip_id", "obs_id"]
data["min_distance"] = data.groupby(group_cols)["distance_m"].transform("min")
data["distance_from_best"] = data["distance_m"] - data["min_distance"]
data["candidate_count"] = data.groupby(group_cols)["edge_id"].transform("size")

FEATURES = [
    "distance_m",
    "distance_rank",
    "min_distance",
    "distance_from_best",
    "candidate_count"
]

data[FEATURES] = data[FEATURES].replace(
    [np.inf, -np.inf], np.nan
)
data[FEATURES] = data[FEATURES].fillna(
    data[FEATURES].median()
)

# Only observations whose correct edge is present can train
# a meaningful candidate-ranking classifier.
has_truth = data.groupby(group_cols)["label"].transform("max").eq(1)

train_data = data[
    (data["split"] == "train") & has_truth
].copy()

student_all = data[data["split"] == "student"].copy()
student_data = student_all[
    student_all.groupby(group_cols)["label"].transform("max").eq(1)
].copy()

if train_data.empty:
    raise ValueError(
        "No training observations contain their reference edge. "
        "Check the edge_id and ref_edge_id formats."
    )

if train_data["label"].nunique() < 2:
    raise ValueError("Training labels contain only one class.")

print("Training candidate rows:", len(train_data))
print("Training observations with a matching candidate:",
      train_data.groupby(group_cols).ngroups)
print("Student candidate rows:", len(student_all))
print("Student observations with a matching candidate:",
      student_data.groupby(group_cols).ngroups)

# --------------------------------------------------
# 3. Metrics helper
# --------------------------------------------------
def evaluate(model, frame, model_name, result_name):
    if frame.empty:
        return {
            "model": model_name,
            "evaluation": result_name,
            "candidate_accuracy": np.nan,
            "candidate_precision": np.nan,
            "candidate_recall": np.nan,
            "candidate_f1": np.nan,
            "top1_accuracy": np.nan,
            "evaluated_observations": 0
        }

    X = frame[FEATURES]
    y = frame["label"].to_numpy()
    probs = model.predict_proba(X)[:, 1]

    scored = frame[group_cols + ["label"]].copy()
    scored["score"] = probs

    # Select the highest-scoring edge for each observation.
    chosen = scored.loc[
        scored.groupby(group_cols)["score"].idxmax()
    ]

    return {
        "model": model_name,
        "evaluation": result_name,
        "candidate_accuracy": accuracy_score(
            y, (probs >= 0.5).astype(int)
        ),
        "candidate_precision": precision_score(
            y, (probs >= 0.5).astype(int), zero_division=0
        ),
        "candidate_recall": recall_score(
            y, (probs >= 0.5).astype(int), zero_division=0
        ),
        "candidate_f1": f1_score(
            y, (probs >= 0.5).astype(int), zero_division=0
        ),
        "top1_accuracy": float(chosen["label"].mean()),
        "evaluated_observations": len(chosen)
    }

# Coverage: proportion of observations whose correct edge
# appears somewhere in the candidate list.
def coverage(frame):
    if frame.empty:
        return 0.0
    return float(
        frame.groupby(group_cols)["label"].max().mean()
    )

# --------------------------------------------------
# 4. Define models
# --------------------------------------------------
def make_model(name):
    if name == "Random Forest":
        return RandomForestClassifier(
            n_estimators=40,
            max_depth=16,
            min_samples_leaf=2,
            n_jobs=-1,
            random_state=42,
            class_weight="balanced_subsample"
        )

    return XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=1.0,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        n_jobs=-1,
        random_state=42
    )

# --------------------------------------------------
# 5. Three-fold cross-validation, then student evaluation
# --------------------------------------------------
results_by_model = {}
scores_by_model = {}

cv = GroupKFold(n_splits=3)

for model_name, output_prefix in [
    ("Random Forest", "random_forest"),
    ("XGBoost", "xgboost")
]:
    print("\n" + "=" * 55)
    print("MODEL:", model_name)
    print("=" * 55)

    metrics = []
    started = time.time()

    X = train_data[FEATURES]
    y = train_data["label"]
    groups = train_data["trip_id"]

    for fold, (tr_idx, va_idx) in enumerate(
        cv.split(X, y, groups), start=1
    ):
        print(f"{model_name}: training CV fold {fold}/3...")
        model = make_model(model_name)

        model.fit(X.iloc[tr_idx], y.iloc[tr_idx])

        fold_result = evaluate(
            model,
            train_data.iloc[va_idx],
            model_name,
            f"CV fold {fold}"
        )
        fold_result["candidate_coverage"] = coverage(
            data[
                (data["split"] == "train")
                & data["trip_id"].isin(
                    train_data.iloc[va_idx]["trip_id"].unique()
                )
            ]
        )
        metrics.append(fold_result)
        print("Fold top-1 accuracy:", fold_result["top1_accuracy"])

    # Fit final model using all eligible training observations.
    print(f"{model_name}: fitting final model...")
    final_model = make_model(model_name)
    final_model.fit(train_data[FEATURES], train_data["label"])

    # Evaluate the held-out student split where the true edge
    # is present among the candidates.
    student_result = evaluate(
        final_model,
        student_data,
        model_name,
        "Student test"
    )
    student_result["candidate_coverage"] = coverage(student_all)
    metrics.append(student_result)

    metrics_file = f"{output_prefix}_metrics.csv"
    pd.DataFrame(metrics).to_csv(metrics_file, index=False)

    # Score every student candidate, including observations
    # where the true edge is absent from the candidate list.
    print(f"{model_name}: scoring student candidates...")
    student_scores = student_all[
        ["trip_id", "obs_id", "edge_id",
         "distance_m", "distance_rank"]
    ].copy()

    student_scores["predicted_score"] = final_model.predict_proba(
        student_all[FEATURES]
    )[:, 1]

    student_scores["predicted_rank"] = (
        student_scores.groupby(["trip_id", "obs_id"])[
            "predicted_score"
        ].rank(method="first", ascending=False).astype(int)
    )

    student_scores = student_scores.sort_values(
        ["trip_id", "obs_id", "predicted_rank"]
    )

    score_file = f"{output_prefix}_student_scores.csv"
    student_scores.to_csv(score_file, index=False)

    results_by_model[model_name] = metrics
    scores_by_model[model_name] = score_file

    print("Saved:", metrics_file)
    print("Saved:", score_file)
    print(f"Elapsed time: {(time.time() - started) / 60:.1f} minutes")

# --------------------------------------------------
# 6. Show and download all four output files
# --------------------------------------------------
print("\nFINAL METRICS")
display(pd.concat([
    pd.read_csv("random_forest_metrics.csv"),
    pd.read_csv("xgboost_metrics.csv")
], ignore_index=True))

output_zip = "PathSnap_Person2_Results.zip"

with zipfile.ZipFile(output_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for name in [
        "random_forest_metrics.csv",
        "xgboost_metrics.csv",
        "random_forest_student_scores.csv",
        "xgboost_student_scores.csv"
    ]:
        z.write(name)

print("\nAll four CSV files are ready.")
files.download(output_zip)

Loading CSV files...
Candidates: (1198560, 5)
Observations: (227577, 4)
Split counts:
split
train      208417
student     19160
Name: count, dtype: int64
Training candidate rows: 1051924
Training observations with a matching candidate: 184366
Student candidate rows: 113648
Student observations with a matching candidate: 19104

MODEL: Random Forest
Random Forest: training CV fold 1/3...
Fold top-1 accuracy: 0.586928546150629
Random Forest: training CV fold 2/3...
Fold top-1 accuracy: 0.5609281518793985
Random Forest: training CV fold 3/3...
Fold top-1 accuracy: 0.6007231174396286
Random Forest: fitting final model...
Random Forest: scoring student candidates...
Saved: random_forest_metrics.csv
Saved: random_forest_student_scores.csv
Elapsed time: 4.6 minutes

MODEL: XGBoost
XGBoost: training CV fold 1/3...
Fold top-1 accuracy: 0.5932020718720844
XGBoost: training CV fold 2/3...
Fold top-1 accuracy: 0.5619465003364187
XGBoost: training CV fold 3/3...
Fold top-1 accuracy: 0.60112484935053

,model,evaluation,candidate_accuracy,candidate_precision,candidate_recall,candidate_f1,top1_accuracy,evaluated_observations,candidate_coverage
0,Random Forest,CV fold 1,0.827360,0.506849,0.908246,0.650619,0.586929,62166,0.994672
1,Random Forest,CV fold 2,0.853239,0.519876,0.893037,0.657179,0.560928,54991,0.911292
2,Random Forest,CV fold 3,0.834083,0.538147,0.931854,0.682278,0.600723,67209,0.997995
3,Random Forest,Student test,0.839700,0.513603,0.921954,0.659700,0.565745,19104,0.997077
4,XGBoost,CV fold 1,0.850281,0.562900,0.689332,0.619733,0.593202,62166,0.994672
5,XGBoost,CV fold 2,0.861890,0.561518,0.562274,0.561896,0.561947,54991,0.911292
6,XGBoost,CV fold 3,0.847335,0.601066,0.598982,0.600022,0.601125,67209,0.997995
7,XGBoost,Student test,0.852165,0.560910,0.565431,0.563161,0.559464,19104,0.997077



All four CSV files are ready.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:

import os

print("Files currently in Colab:")
for f in os.listdir("/content"):
    print(f)


Files currently in Colab:
.config
candidates.csv
Student folder-20261118T050914Z-1-001.zip
observations (1).csv
observations.csv
Student folder.zip
sample_data


In [9]:

import os

# Use the filenames that actually exist in Colab
ZIP_FILE = "Student folder.zip"
OBS_FILE = "observations.csv"

print("Candidates:", os.path.exists("candidates.csv"))
print("Observations:", os.path.exists(OBS_FILE))
print("Student ZIP:", os.path.exists(ZIP_FILE))


Candidates: True
Observations: True
Student ZIP: True
